In [6]:
# 02_baseline.ipynb — Baseline Model Training

# Train a simple unimodal classifier (MLP / Logistic Regression)
# on the preprocessed PBMC3k dataset to establish baseline metrics.

In [10]:
import os

# Find project root dynamically
while not os.path.exists("requirements.txt") and os.path.dirname(os.getcwd()) != "/":
    os.chdir("..")

# confirm working directory
print("Working directory set to:", os.getcwd())

# define artifact directory relative to project root
ARTIFACTS_DIR = os.path.join(os.getcwd(), "artifacts")
print("Artifacts dir contents:", os.listdir(ARTIFACTS_DIR))

Working directory set to: /Users/anuragdani/Github/neuroai-project
Artifacts dir contents: ['splits.json', '.keep', 'qc_summary.csv', 'preprocessed_small.h5ad']


In [11]:
import anndata as ad
import json
import numpy as np

adata_path = os.path.join(ARTIFACTS_DIR, "preprocessed_small.h5ad")
splits_path = os.path.join(ARTIFACTS_DIR, "splits.json")

print("Reading:", adata_path)
adata = ad.read_h5ad(adata_path)

with open(splits_path) as f:
    splits = json.load(f)

X = adata.X.A if hasattr(adata.X, "A") else adata.X
y = adata.obs["label"].astype("category").cat.codes.values


Reading: /Users/anuragdani/Github/neuroai-project/artifacts/preprocessed_small.h5ad


In [12]:
import os
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
import os, json, numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import anndata as ad

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)


Using device: mps


In [13]:
adata = ad.read_h5ad(os.path.join(ARTIFACTS_DIR, "preprocessed_small.h5ad"))
with open(os.path.join(ARTIFACTS_DIR, "splits.json")) as f:
    splits = json.load(f)

X = adata.X.A if hasattr(adata.X, "A") else adata.X
y = adata.obs["label"].astype("category").cat.codes.values

train_idx, val_idx, test_idx = (
    np.array(splits["train_idx"]),
    np.array(splits["val_idx"]),
    np.array(splits["test_idx"]),
)

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (1688, 1838), Val: (422, 1838), Test: (528, 1838)


In [14]:
batch_size = 128

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                  torch.tensor(y_train, dtype=torch.long)),
    batch_size=batch_size, shuffle=True)

val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                  torch.tensor(y_val, dtype=torch.long)),
    batch_size=batch_size)

test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                  torch.tensor(y_test, dtype=torch.long)),
    batch_size=batch_size)


In [15]:
input_dim = X_train.shape[1]
n_classes = len(np.unique(y))
print(f"Input dim: {input_dim}, Classes: {n_classes}")

class MLP(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes)
        )
    def forward(self, x): return self.net(x)

model = MLP(input_dim, n_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


Input dim: 1838, Classes: 6


In [16]:
n_epochs = 10
best_val = 0.0

for epoch in range(n_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    # Validation phase
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            out = model(xb.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())

    acc = accuracy_score(trues, preds)
    print(f"Epoch {epoch+1}/{n_epochs} — Val Acc: {acc:.4f}")

    if acc > best_val:
        best_val = acc
        torch.save(model.state_dict(), os.path.join(ARTIFACTS_DIR, "baseline_mlp.pt"))


Epoch 1/10 — Val Acc: 0.1635
Epoch 2/10 — Val Acc: 0.4479
Epoch 3/10 — Val Acc: 0.4479
Epoch 4/10 — Val Acc: 0.4479
Epoch 5/10 — Val Acc: 0.4479
Epoch 6/10 — Val Acc: 0.4479
Epoch 7/10 — Val Acc: 0.4479
Epoch 8/10 — Val Acc: 0.4479
Epoch 9/10 — Val Acc: 0.4479
Epoch 10/10 — Val Acc: 0.4479


In [ ]:
# --- Load best model and evaluate on test set ---
model.load_state_dict(torch.load(os.path.join(ARTIFACTS_DIR, "baseline_mlp.pt")))
model.eval()

preds, probs, trues = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        probs.extend(torch.softmax(out, 1)[:, 1].cpu().numpy() if n_classes == 2
                     else out.softmax(1).max(1).values.cpu().numpy())
        trues.extend(yb.numpy())

acc = accuracy_score(trues, preds)
f1  = f1_score(trues, preds, average="weighted")
try:
    auc = roc_auc_score(trues, probs, multi_class="ovo")
except Exception:
    auc = float("nan")

metrics = {"accuracy": acc, "f1_weighted": f1, "auc": auc}
pd.DataFrame([metrics]).to_csv(os.path.join(ARTIFACTS_DIR, "baseline_metrics.csv"), index=False)
print("Baseline metrics:", metrics)


✅ Baseline metrics: {'accuracy': 0.44696969696969696, 'f1_weighted': 0.2761383468189751, 'auc': nan}
